In [1]:
%pip intall duckdb

Note: you may need to restart the kernel to use updated packages.


ERROR: unknown command "intall" - maybe you meant "install"



In [26]:
from pathlib import Path
import duckdb
import pandas as pd

print("DuckDB", duckdb.__version__)

DuckDB 1.5.5


In [27]:


pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


RUTA_PARQUET = Path("ene_cruda.parquet")
PARQ = RUTA_PARQUET.as_posix()



# DuckDB en memoria; NO carga el Parquet a RAM, lo consulta desde disco.
con = duckdb.connect()
# En SQL usamos '/' aunque estés en Windows (evita problemas con '\').
PARQ = RUTA_PARQUET.as_posix()




In [31]:
serie = con.sql(f"""
with base as (
    select
        cast(ano_trimestre as int)                         as annio,
        cast(mes_central  as int)                          as mes,
        coalesce(
            cast(region_15 as int),
            case when cast(region as int) = 16 then 8      -- Ñuble vuelve a Biobío
                 else cast(region as int) end
        )                                                  as region_15h,
        activ,
        cast(replace(fact_cal, ',', '.') as double)        as fact
    from '{PARQ}'
),
regional as (
    select
        annio,
        mes,
        region_15h                                         as region_15,
        sum(fact)                                          as poblacion,
        sum(case when activ = '1' then fact else 0 end)    as ocupados,
        sum(case when activ = '2' then fact else 0 end)    as desocupados
    from base
    group by all
),
nacional as (
    select
        annio,
        mes,
        99                                                 as region_15,
        sum(poblacion)                                     as poblacion,
        sum(ocupados)                                      as ocupados,
        sum(desocupados)                                   as desocupados
    from regional
    group by all
),
unido as (
    select * from regional
    union all
    select * from nacional
)
select
    annio,
    mes,
    make_date(annio, mes, 1)                                 as fecha,
    region_15,
    poblacion,
    ocupados,
    desocupados,
    desocupados / (ocupados + desocupados)                 as tasa_desempleo
from unido
order by fecha, region_15
""").df()
serie.to_csv("serie_desempleo_regional.csv", index=False, encoding="utf-8-sig")
serie

,annio,mes,fecha,region_15,poblacion,ocupados,desocupados,tasa_desempleo
0,2010,2,2010-02-01,1,"291,760.000","135,903.591","10,037.281",0.069
1,2010,2,2010-02-01,2,"559,452.001","238,440.263","22,602.339",0.087
2,2010,2,2010-02-01,3,"284,706.000","117,058.364","11,780.992",0.091
3,2010,2,2010-02-01,4,"702,640.000","284,381.615","32,324.298",0.102
4,2010,2,2010-02-01,5,"1,739,119.000","712,319.152","89,748.022",0.112
...,...,...,...,...,...,...,...,...
3131,2026,5,2026-05-01,12,"185,048.000","99,398.799","7,315.215",0.069
3132,2026,5,2026-05-01,13,"8,513,450.000","4,203,030.580","435,622.375",0.094
3133,2026,5,2026-05-01,14,"415,558.000","182,570.389","20,665.589",0.102
3134,2026,5,2026-05-01,15,"265,298.000","119,302.097","12,734.346",0.096
